In [1]:
import numpy as np
import pandas as pd
import glob
import geopy
import geopy.distance
import datetime
import sys
import time

In [ ]:
path = r"/content/drive/My Drive/Colab Notebooks/Ship_data.csv"
df=pd.read_csv(path,sep=',',low_memory=False)
df

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
#Delete features that wont be needed for triaining



del df['mmsi']




In [7]:
df

,Length,Breadth,Draught,Longitude,Latitude,SOG,bool,Hour
0,152,24,6.8,5.838777,53.63944,14.3,0,12.685507
1,152,24,6.8,5.845800,53.64076,14.3,0,12.667177
2,152,24,6.8,5.852149,53.64198,14.3,0,12.650496
3,152,24,6.8,5.858515,53.64320,14.3,0,12.633836
4,152,24,6.8,5.864876,53.64441,14.3,0,12.617179
...,...,...,...,...,...,...,...,...
266885,229,38,13.3,9.806900,53.55355,7.4,1,0.083519
266886,229,38,13.3,9.810224,53.55293,7.3,1,0.066774
266887,229,38,13.3,9.813442,53.55230,7.2,1,0.049974
266888,229,38,13.3,9.816650,53.55172,7.1,1,0.033443


In [8]:
#Training part of the data
from sklearn import metrics

from pandas.plotting import scatter_matrix
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

from hyperopt import fmin, tpe, hp,STATUS_OK, Trials
import matplotlib.pyplot as plt
!pip install tpot
from tpot import TPOTClassifier

     |████████████████████████████████| 92kB 3.7MB/s 
     |████████████████████████████████| 163kB 16.0MB/s 
  Created wheel for stopit: filename=stopit-1.1.2-cp36-none-any.whl size=11956 sha256=e82499fa3da337960d8786e9e46ffaa246fbd45098c02bf5afe51356997265c5
  Stored in directory: /root/.cache/pip/wheels/3c/85/2b/2580190404636bfc63e8de3dff629c03bb795021e1983a6cc7
Successfully built stopit


In [10]:
#Features
features = df.drop('bool', axis=1)
features= np.array(features)

# Labels
labels= np.array(df['bool'])

feature_list = list(df.columns)

In [11]:
#Splitting the data to train and test

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size = 0.25, random_state = 42)

In [12]:
print('Training Features Shape:', X_train.shape)
print('Training Labels Shape:', y_train.shape)
print('Testing Features Shape:', X_test.shape)
print('Testing Labels Shape:',  y_test.shape)



Training Features Shape: (200167, 7)
Training Labels Shape: (200167,)
Testing Features Shape: (66723, 7)
Testing Labels Shape: (66723,)


In [17]:
# Training part with logistic regression without tuning(Default values)

lr = LogisticRegression()

start_time = time.time()

lr.fit(X_train, y_train)

sv_notune= time.time() - start_time

y_pred= lr.predict(X_test)


print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

Accuracy: 0.7913762870374533


/usr/local/lib/python3.6/dist-packages/sklearn/linear_model/_logistic.py:940: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG)


In [18]:
# SVM object to be used to tune the default kernel will be 'rbf' 

lrc = LogisticRegression()



In [21]:
lrc.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': None,
 'max_iter': 100,
 'multi_class': 'auto',
 'n_jobs': None,
 'penalty': 'l2',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [23]:
# Tuning LRC with the following algorithims
        # 1) Random
        # 2) Bayesian
        #  3) Genetic algorithims
        
# Random searcch set parameters
C=np.random.uniform(0.01,100,40)
max_iter = np.random.randint(100,1000,40)

random_grid = {'C': C, 
               'max_iter':max_iter
               }
random_grid 

{'C': array([96.23132697, 88.42835642, 68.09509226, 71.04753576, 36.15921613,
        81.61173625, 30.96871399, 40.45084022, 27.32768988, 44.3356856 ,
        87.52031497, 72.78560164, 39.13861329, 17.69464815, 52.19449008,
        93.19185675, 85.06557477, 69.78117003,  8.21925473,  6.49664747]),
 'max_iter': array([650, 297, 386, 408, 303, 433, 754, 543, 854, 278,  64, 756, 935,
        555, 819, 133, 358, 176, 416, 294])}

In [24]:
#Random search tuning
lr_random = RandomizedSearchCV(estimator = lrc, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=10, random_state=42, n_jobs =-1)
start_time = time.time()
lr_random.fit(X_train, y_train)

random_time =time.time() - start_time

Fitting 3 folds for each of 100 candidates, totalling 300 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    5.8s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:   12.4s
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:   17.4s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   24.0s
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:   31.3s
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:   41.2s
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   52.4s
[Parallel(n_jobs=-1)]: Done  57 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done  68 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done  94 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 109 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 124 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:  2

In [25]:
#Parameters selected for random search
lr_random.get_params()


{'cv': 3,
 'error_score': nan,
 'estimator': LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                    intercept_scaling=1, l1_ratio=None, max_iter=100,
                    multi_class='auto', n_jobs=None, penalty='l2',
                    random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                    warm_start=False),
 'estimator__C': 1.0,
 'estimator__class_weight': None,
 'estimator__dual': False,
 'estimator__fit_intercept': True,
 'estimator__intercept_scaling': 1,
 'estimator__l1_ratio': None,
 'estimator__max_iter': 100,
 'estimator__multi_class': 'auto',
 'estimator__n_jobs': None,
 'estimator__penalty': 'l2',
 'estimator__random_state': None,
 'estimator__solver': 'lbfgs',
 'estimator__tol': 0.0001,
 'estimator__verbose': 0,
 'estimator__warm_start': False,
 'iid': 'deprecated',
 'n_iter': 100,
 'n_jobs': -1,
 'param_distributions': {'C': array([96.23132697, 88.42835642, 68.09509226, 71.04753576, 36.15921613,
         81.6

In [ ]:
#Tuning with Bayesian optimization

search_space= {'C': hp.uniform ('C', 0.01, 100),
        'max_iter': hp.uniform ('max_iter', 100, 1000)
    }
def objective(search_space):
    bayes = LogisticRegression(C = search_space['C'],max_iter = search_space['max_iter'])
    accuracy = cross_val_score(bayes, X_train, y_train, cv = 4).mean()
    return {'loss': accuracy, 'status': STATUS_OK }

trials = Trials()


# Training from bayes tuning


start_time = time.time()


best = fmin(fn= objective,
            space= search_space,
            algo= tpe.suggest,
            max_evals = 50,
            trials= trials)
best



trainedlrc  = LogisticRegression(C = best['C'],max_iter = best['max_iter']).fit(X_train, y_train)

bayesian_time =time.time() - start_time

y_pred2 = trainedlrc.predict(X_test)

 34%|███▍      | 17/50 [02:37<04:40,  8.50s/it, best loss: 0.7900902694385544]

In [37]:
# The genetic algorithim part

lrc_tpot = TPOTClassifier(verbosity=3, 
 scoring='accuracy', 
 random_state=32, 
 n_jobs=-1, 
 generations=5, 
 population_size=10,
 early_stop=5)

start_time = time.time()

lrc_tpot.fit(X_train, y_train) 

genetic_time = time.time() - start_time

32 operators have been imported by TPOT.


Skipped pipeline #2 due to time out. Continuing to the next pipeline.
_pre_test decorator: _random_mutation_operator: num_test=0 Unsupported set of arguments: The combination of penalty='l1' and loss='hinge' is not supported, Parameters: penalty='l1', loss='hinge', dual=False.
Pipeline encountered that has previously been evaluated during the optimization process. Using the score from the previous evaluation.

Generation 1 - Current Pareto front scores:
-1	0.9801515675430785	ExtraTreesClassifier(input_matrix, ExtraTreesClassifier__bootstrap=True, ExtraTreesClassifier__criterion=entropy, ExtraTreesClassifier__max_features=0.8, ExtraTreesClassifier__min_samples_leaf=19, ExtraTreesClassifier__min_samples_split=2, ExtraTreesClassifier__n_estimators=100)
Generation 2 - Current Pareto front scores:
-1	0.9801515675430785	ExtraTreesClassifier(input_matrix, ExtraTreesClassifier__bootstrap=True, ExtraTreesClassifier__criterion=entropy, ExtraTreesClassifier__max_features=0.8, ExtraTreesClassifier

In [43]:
lrc_tpot.fitted_pipeline_

Pipeline(memory=None,
         steps=[('extratreesclassifier',
                 ExtraTreesClassifier(bootstrap=True, ccp_alpha=0.0,
                                      class_weight=None, criterion='entropy',
                                      max_depth=None,
                                      max_features=0.7500000000000001,
                                      max_leaf_nodes=None, max_samples=None,
                                      min_impurity_decrease=0.0,
                                      min_impurity_split=None,
                                      min_samples_leaf=20, min_samples_split=9,
                                      min_weight_fraction_leaf=0.0,
                                      n_estimators=100, n_jobs=None,
                                      oob_score=False, random_state=32,
                                      verbose=0, warm_start=False))],
         verbose=False)

In [39]:
# Results of training (time and classification report)

#1 Result of LRC without tuning
print("%s seconds it took to train without tuning" % (sv_notune))    
print(classification_report(y_test,y_pred))

#2 Result of LRC with Random search tuning
print("%s seconds it took to train with Random search tuning" % (random_time))
print(classification_report(y_test,y_pred1))

#3 Result of LRC with Bayesian search tuning 
print("%s seconds it took to train with Bayesian search tuning " % (bayesian_time))
print(classification_report(y_test,y_pred2))

#4 Result of Classification with Genetic Algorithim
print("%s seconds it took to train genetic algorithim" % (genetic_time))
print(classification_report(y_test,predictions3))


3.3291521072387695 seconds it took to train without tuning
              precision    recall  f1-score   support

           0       0.79      0.99      0.88     52368
           1       0.67      0.06      0.11     14355

    accuracy                           0.79     66723
   macro avg       0.73      0.53      0.50     66723
weighted avg       0.77      0.79      0.72     66723

635.3997950553894 seconds it took to train with Random search tuning
              precision    recall  f1-score   support

           0       0.80      0.99      0.88     52368
           1       0.70      0.07      0.13     14355

    accuracy                           0.79     66723
   macro avg       0.75      0.53      0.51     66723
weighted avg       0.77      0.79      0.72     66723

502.2593398094177 seconds it took to train with Bayesian search tuning 
              precision    recall  f1-score   support

           0       0.79      0.99      0.88     52368
           1       0.61      0.04    